# 01 · DDPM 前向过程与加噪  DDPM Forward Process & Noise Schedule

> **能力标签**：GA3（扩散模型与流匹配 / Diffusion Models & Flow Matching）

## 目标 Objectives

完成本课后，你应该能够：

1. 描述 **DDPM（Denoising Diffusion Probabilistic Models）前向过程**（forward process）：如何在 T 步内逐步将数据 $x_0$ 加噪为接近高斯噪声的 $x_T$。
2. 推导**噪声表**（noise schedule）：线性 $\beta_t$ 序列及 $\alpha_t = 1 - \beta_t$、$\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$。
3. 写出**前向采样**（forward sampling）的闭合形式（closed form）：
   $q(x_t | x_0) = \mathcal{N}(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I)$
4. 实现 `linear_beta_schedule` 和 `q_sample`，在合成数据上验证端点性质。
5. 解释**去噪目标**（denoising target）：训练神经网络预测加入的噪声 $\epsilon$（predict-$\epsilon$ 参数化）。

> 对应能力：**GA3**

## 概念讲解 Concepts

### 1. DDPM 前向过程 Forward Process

DDPM（Ho et al., 2020）定义一个**马尔科夫链**（Markov chain），在 $T$ 步内将数据逐步"腐化"为高斯噪声：

$$q(x_1, \ldots, x_T | x_0) = \prod_{t=1}^{T} q(x_t | x_{t-1})$$

每步加噪核（transition kernel）：

$$q(x_t | x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right)$$

其中 $\beta_t \in (0, 1)$ 是**噪声强度**（noise level），随步数单调递增。

---

### 2. 噪声表 Noise Schedule

**线性噪声表**（linear schedule）：$\beta_t$ 从 $\beta_1 = 10^{-4}$ 线性增长到 $\beta_T = 0.02$。

定义辅助量：

| 符号 | 定义 | 含义 |
|------|------|------|
| $\alpha_t$ | $1 - \beta_t$ | 单步保留比例 signal retention |
| $\bar{\alpha}_t$ | $\prod_{s=1}^{t} \alpha_s$ | 累积保留比例 cumulative retention |

随着 $t$ 增大，$\bar{\alpha}_t \to 0$，信号完全淹没于噪声。

---

### 3. 前向采样闭合形式 Closed-Form Forward Sample

利用高斯分布的可加性，可以**一步直达**任意时间步 $t$：

$$q(x_t | x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1 - \bar{\alpha}_t) I\right)$$

等价于采样公式（`q_sample`）：

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

这是 DDPM 训练的核心：无需逐步迭代，直接从 $x_0$ 得到任意噪声水平的 $x_t$。

**端点验证**（endpoint checks）：
- $t = 0$：$\bar{\alpha}_0 = 1 \Rightarrow x_0 = x_0$（无噪声）
- $t = T$：$\bar{\alpha}_T \approx 0 \Rightarrow x_T \approx \epsilon$（纯噪声）

---

### 4. 去噪目标 Denoising Objective

DDPM 训练的神经网络 $\epsilon_\theta(x_t, t)$ 预测加入的噪声 $\epsilon$，即**predict-$\epsilon$ 参数化**：

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\!\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

其中 $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$。

这个目标等价于在每个时间步训练一个**去噪自编码器**（denoising autoencoder），但所有时间步共享参数 $\theta$，并用时间步嵌入（time embedding）区分不同噪声水平。

---

### 5. 实现摘要 Implementation Summary

```python
# 线性噪声表 Linear noise schedule
betas = torch.linspace(beta_start, beta_end, T)       # (T,)
alphas = 1.0 - betas                                   # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)              # (T,)

# 前向采样 Forward sampling (q_sample)
ab = alpha_bars[t].unsqueeze(1)                        # (B, 1)
x_t = sqrt(ab) * x0 + sqrt(1 - ab) * noise            # (B, D)
```

## 示例 Worked Example

In [ ]:
# Setup
import math
import tempfile
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)
DEVICE = torch.device('cpu')
TMPDIR = tempfile.mkdtemp()
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Tmp dir:', TMPDIR)

In [ ]:
# linear_beta_schedule + q_sample (mirrors w9-ddpm)

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    '''线性 beta 噪声表，返回 (betas, alphas, alpha_bars)，每个长度 T。
    Linear beta noise schedule; returns (betas, alphas, alpha_bars) each of length T.
    '''
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars


def q_sample(x0, t, noise, alpha_bars):
    '''前向加噪：x_t = sqrt(alpha_bar_t) x0 + sqrt(1-alpha_bar_t) noise。
    Forward noising (closed form): x_t = sqrt(ab_t)*x0 + sqrt(1-ab_t)*noise.
    x0, noise: (B, D); t: (B,) long index tensor.
    '''
    ab = alpha_bars[t].unsqueeze(1)          # (B, 1)
    return torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * noise


# --- 单元测试 Unit Tests ---
T = 1000
betas, alphas, alpha_bars = linear_beta_schedule(T)

print(f'betas:      [{betas[0]:.6f}, ..., {betas[-1]:.6f}]')
print(f'alpha_bars: [{alpha_bars[0]:.6f}, ..., {alpha_bars[-1]:.6f}]')

# 端点验证 endpoint checks
assert abs(betas[0].item() - 1e-4) < 1e-7, 'beta_start 错误'
assert abs(betas[-1].item() - 0.02) < 1e-7, 'beta_end 错误'
assert alpha_bars[0].item() > 0.99, 'alpha_bar[0] 应接近 1'
assert alpha_bars[-1].item() < 0.02, 'alpha_bar[T-1] 应接近 0'

# q_sample 端点 q_sample endpoint
torch.manual_seed(0)
B, D = 8, 4
x0 = torch.randn(B, D)
noise = torch.randn(B, D)

# t=0: x_t 应接近 sqrt(ab[0]) * x0
t0 = torch.zeros(B, dtype=torch.long)
xt0 = q_sample(x0, t0, noise, alpha_bars)
expected0 = alpha_bars[0].sqrt() * x0 + (1 - alpha_bars[0]).sqrt() * noise
assert torch.allclose(xt0, expected0, atol=1e-5), 'q_sample t=0 错误'

# t=T-1: 几乎纯噪声
tT = torch.full((B,), T - 1, dtype=torch.long)
xtT = q_sample(x0, tT, noise, alpha_bars)
noise_ratio = (xtT - xtT.mean(0)).norm() / (noise - noise.mean(0)).norm()
print(f'q_sample(t=T-1) noise ratio: {noise_ratio.item():.4f}  (接近 1 = 纯噪声)')
assert noise_ratio.item() > 0.9, '末端应接近纯噪声'

print('\u2713 linear_beta_schedule 通过 passed')
print('\u2713 q_sample 通过 passed')

In [ ]:
# 逐步前向加噪可视化 Visualize forward noising process

torch.manual_seed(42)
T = 1000
betas, alphas, alpha_bars = linear_beta_schedule(T)

D = 2  # 2D for easy visualization
x0_vis = torch.tensor([[2.0, 1.0], [-1.5, 2.0], [0.5, -2.0]])  # 3 points
B_vis = x0_vis.shape[0]

timesteps = [0, 50, 200, 500, 800, 999]
results = {}
for t_val in timesteps:
    t_idx = torch.full((B_vis,), t_val, dtype=torch.long)
    noise = torch.randn_like(x0_vis)
    xt = q_sample(x0_vis, t_idx, noise, alpha_bars)
    results[t_val] = xt.clone()

# Print alpha_bar values at selected timesteps
print('时间步 alpha_bar 值 (alpha_bar at selected timesteps):')
for t_val in timesteps:
    print(f'  t={t_val:4d}: alpha_bar={alpha_bars[t_val]:.4f}  '
          f'signal={alpha_bars[t_val].sqrt():.4f}  '
          f'noise={(1 - alpha_bars[t_val]).sqrt():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: forward noising trajectory for first point
x_vals = [results[t][0, 0].item() for t in timesteps]
y_vals = [results[t][0, 1].item() for t in timesteps]
axes[0].plot(x_vals, y_vals, 'o-', alpha=0.7, color='steelblue')
axes[0].scatter([x0_vis[0, 0]], [x0_vis[0, 1]], s=100, color='red',
                zorder=5, label='$x_0$')
axes[0].scatter([x_vals[-1]], [y_vals[-1]], s=100, color='gray',
                zorder=5, label='$x_{T-1}$')
for i, t_val in enumerate(timesteps):
    axes[0].annotate(f't={t_val}', (x_vals[i], y_vals[i]),
                     textcoords='offset points', xytext=(5, 5), fontsize=7)
axes[0].set_title('前向加噪轨迹 Forward Noising Trajectory')
axes[0].set_xlabel('Dim 0'); axes[0].set_ylabel('Dim 1')
axes[0].legend()

# Plot 2: alpha_bar decay curve
t_range = torch.arange(T)
axes[1].plot(t_range.numpy(), alpha_bars.numpy(), color='coral', linewidth=2)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\bar{\alpha}_t$')
axes[1].set_title(r'累积噪声表 Cumulative Noise Schedule $\bar{\alpha}_t$')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(TMPDIR, 'ddpm_forward.png')
plt.savefig(fig_path, dpi=80); plt.close()
print(f'图像已保存 Saved: {fig_path}')

## 动手 Exercise

实现 `ddpm_denoising_loss`：给定 $x_0$，随机采样时间步 $t$ 和噪声 $\epsilon$，用 `q_sample` 计算 $x_t$，再用简单网络预测噪声，返回 MSE 损失。


In [ ]:
# Exercise: DDPM denoising loss training loop

import torch.nn as nn
import torch.nn.functional as F

def ddpm_denoising_loss(model, x0, alpha_bars):
    '''DDPM 去噪训练步骤 (predict-epsilon 参数化).
    随机采样 t 和 epsilon，用 q_sample 得到 x_t，让 model 预测 epsilon。
    Randomly sample t and epsilon; q_sample to get x_t; model predicts epsilon.
    Args:
        model: 网络 f(x_t, t_emb) -> epsilon_hat，输入 (B, D+1)，输出 (B, D)
        x0: (B, D)
        alpha_bars: (T,)
    Returns:
        scalar MSE loss
    '''
    # TODO: implement
    # Hint:
    #   T_len = alpha_bars.shape[0]
    #   B = x0.shape[0]
    #   t = torch.randint(0, T_len, (B,))
    #   noise = torch.randn_like(x0)
    #   x_t = q_sample(x0, t, noise, alpha_bars)
    #   t_emb = (t.float() / T_len).unsqueeze(1)
    #   inp = torch.cat([x_t, t_emb], dim=1)
    #   noise_pred = model(inp)
    #   return F.mse_loss(noise_pred, noise)
    raise NotImplementedError('请实现 ddpm_denoising_loss')


# Uncomment after implementation:
# T_ex = 100
# _, _, ab_ex = linear_beta_schedule(T_ex)
# net = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))  # D=2
# x0_ex = torch.randn(8, 2)
# loss_ex = ddpm_denoising_loss(net, x0_ex, ab_ex)
# print(f'Loss: {loss_ex.item():.4f}')
# assert loss_ex.item() > 0
# print('\u2713 ddpm_denoising_loss 通过')
print('提示：实现 ddpm_denoising_loss 后取消注释运行验证')

## 延伸阅读 Further Reading

1. **Ho et al. "Denoising Diffusion Probabilistic Models" (NeurIPS 2020)**: <https://arxiv.org/abs/2006.11239> — DDPM 原始论文；第 2 节推导前向过程与 ELBO，第 3 节给出 predict-$\epsilon$ 目标函数。
2. **Sohl-Dickstein et al. "Deep Unsupervised Learning using Nonequilibrium Thermodynamics" (ICML 2015)**: <https://arxiv.org/abs/1503.03585> — 扩散模型的早期工作，奠定数学框架。
3. **Song & Ermon "Generative Modeling by Estimating Gradients of the Data Distribution" (NeurIPS 2019)**: <https://arxiv.org/abs/1907.05600> — 分数匹配（score matching）视角，与 DDPM 密切相关。
4. **Lilian Weng "What are Diffusion Models?" (2021)**: <https://lilianweng.github.io/posts/2021-07-11-diffusion-models/> — 系统性博客综述，推导详尽，强烈推荐配合阅读。
5. **Nichol & Dhariwal "Improved Denoising Diffusion Probabilistic Models" (ICML 2021)**: <https://arxiv.org/abs/2102.09672> — 改进的噪声表与混合目标函数。

## 关联练习 Related Assignments

```bash
python check.py 01-ddpm
```

> 实现 `linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02)` 和 `q_sample(x0, t, noise, alpha_bars)`。
> 关键：`alpha_bars = cumprod(1 - betas)`；`q_sample` 用 `alpha_bars[t].unsqueeze(1)` 广播到 `(B, D)`。

> 能力标签：**GA3** · threshold ≥ 0.7